# KAGGLE 2c: base papers vs ours (comparison)
1) Add Input: the output of 'A3 Track C - Main Part 1'. 2) GPU T4 x2, Internet On. 3) Save Version -> Save & Run All (Commit). Reuses the part 1 models and evaluations (seed 42); adds the MMMC out-of-domain test with a separate 7B judge, significance tests, verdict, tables and figures (~2-2.5 h).

# MG 3010 Assignment 3 — Base papers vs our method (re-implementation and comparison)
Aafreen Zahra Kazmi (26I-7675) · companion to `Assignment3_TrackC_FULL.ipynb` and `Assignment3_Ablation_Study.ipynb`

## The three base papers
| # | Paper | Venue | Link | Role here |
|---|---|---|---|---|
| 1 | X. Chen, B. Zhang, X. Shi, J. Yu, **MC-R1: Mitigating Hallucinations via Reinforcement Learning With String-Match-Based Verifiable Rewards Under Modality Conflicts** | IEEE Trans. Multimedia, vol. 28, pp. 13829–13842, 2026 (Q1) | https://doi.org/10.1109/TMM.2026.3726309 | main base paper (reward-side method) |
| 2 | Z. Wang et al., **Perception-Aware Policy Optimization for Multimodal Reasoning (PAPO)** | ICLR 2026 | https://arxiv.org/abs/2507.06448 | loss-side baseline (masked image) |
| 3 | **CFPO: CounterFactual Policy Optimization** | arXiv 2026 | https://arxiv.org/abs/2606.23206 | loss-side baseline (counterfactual attention) |
| (data) | Z. Zhang et al., **Robust Multimodal LLMs Against Modality Conflict (MMMC)** | ICML 2025 | https://arxiv.org/abs/2507.07151 · data: https://huggingface.co/datasets/MM-Hallu/MMMC | precursor of MC-R1; its test set is our out-of-domain benchmark |

## How each paper works (methodology)
**MC-R1 (reward side).** RLVR with GRPO for open-ended VQA under modality conflict (image and question disagree). Instead of an LLM judge it uses three *string-match* verifiable rewards: (i) an *irrelevant-information-excluded weighted ROUGE-L* in which tokens are weighted by their dependency-parse role and function words are excluded, (ii) a *Key Token Overlap Rate* between answer and reference, and (iii) a *Reasoning Faithfulness Constraint* so the final answer is supported by the reasoning. A new answer `None` is the ground truth for conflict inputs and is penalised on non-conflict inputs. Tested on Qwen2.5-VL, InternVL-3.5 and LLaVA-1.5.
*Re-implementation:* `r_mcr1` (Section 5 of the core code). The three rewards follow the paper's description; the weights in `MCR1` are our reconstruction from the abstract — **check them against the equations in the full text (open access) and edit `MCR1`**.

**PAPO (loss side).** Keeps the GRPO reward unchanged and adds an *implicit perception loss*: maximise $KL[\pi_\theta(o|q,I)\,\|\,\pi_\theta(o|q,I_{mask})]$ where $I_{mask}$ has ~60% of image patches masked, estimated per token with $r-\log r-1$; a *double-entropy* term keeps the KL from being "hacked". Qwen2.5-VL-3B/7B, ViRL39K, lr 1e-6.
*Re-implementation:* `PerceptionGRPOTrainer(mode="papo")` (Section 8b).

**CFPO (loss side).** Also adds a KL regulariser to GRPO, but the corrupted input is a *counterfactual*: inside attention layers, the image values with high text→image attention (above $\mu+2\sigma$) are replaced by the mean image value; the policy is pushed away from this counterfactual prediction ($\gamma=0.02$). Qwen2.5-VL-3B, ViRL39K.
*Re-implementation:* `PerceptionGRPOTrainer(mode="cfpo")` — applied on the visual tokens entering the language model instead of inside every attention layer ("CFPO-style").

**Ours (reward side).** `outcome + A-gated`: format + outcome reward, plus the judge-free, annotation-free consistency reward A (reasoning↔answer consistency, CLIP self-verification of claimed objects, anti-hacking quality gate), applied **only** in GRPO groups whose outcome reward has zero variance.

## Fair-comparison protocol
Same backbone (Qwen2.5-VL-3B), same LoRA, same training data, steps, learning rate, group size and seeds for every method; the only difference is the method's own reward or loss term. Plain GRPO (`outcome` reward) is included so that every method's gain over vanilla RLVR is visible. Evaluation:
* **In-domain:** our COCO modality-conflict test set (accuracy, conflict accuracy, false-None, CHAIR on reasoning, grey-image tests, AURC).
* **Out-of-domain:** MMMC test (MC-R1's own problem setting; ROUGE-L, weighted ROUGE-L, KTOR and a hallucination rate judged by a separate Qwen2.5-VL-7B judge) and POPE.
* **Cost:** GPU hours, peak memory, extra forward passes, and what each method needs (gold answers, annotations, judge).
* **Statistics:** mean ± sd over seeds; ours vs each baseline with paired bootstrap 95% CI and exact McNemar (Holm-corrected). The verdict is computed from the numbers — if a baseline wins on a metric, the notebook says so.

## 0. Install

In [ ]:
# STEP 0 (Kaggle): bring in the part 1 output (found anywhere under /kaggle/input)
import glob, shutil, os
hits = glob.glob("/kaggle/input/**/results/registry.jsonl", recursive=True)
if not hits:
    for r, d, f in os.walk("/kaggle/input"):
        if r.count("/") <= 6: print(r, d[:6], f[:4])
assert hits, "Part 1 output not found: check Add Input and that part 1 has an Output"
src = os.path.dirname(os.path.dirname(hits[0]))          # the a3 folder of part 1
shutil.copytree(src, "/kaggle/working/a3", dirs_exist_ok=True, ignore=shutil.ignore_patterns("coco"))
print("copied from", src); print(sorted(os.listdir("/kaggle/working/a3/runs")))

In [ ]:
import subprocess, sys
def pip(*p): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=False)
# optional, only if CFG["use_vllm"]=True later:  pip("vllm")
pip("trl>=0.21", "peft>=0.13", "bitsandbytes", "transformers>=4.51", "datasets", "accelerate",
    "qwen-vl-utils", "scipy", "matplotlib", "pillow", "pandas", "tabulate", "spacy")
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=False)   # dependency parser for the MC-R1 rewards
# Kaggle ships an old torchao that makes recent PEFT refuse to load LoRA adapters on non-quantized models; we do not need it
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)

## 1. Configuration

In [ ]:
import os
# Kaggle "GPU T4 x2" shows 2 GPUs; the HF Trainer would then wrap the model in DataParallel, which breaks 4-bit + LoRA + generation.
# Use one GPU for everything (must be set before torch is imported).
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import re, json, time, random, math, collections, itertools, zipfile, urllib.request, gc, inspect, shutil, warnings
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

CFG = dict(
    preset="main",                     # "smoke" | "pilot" | "main"
    drive=False,                        # True on Colab: save everything to Google Drive
    model="Qwen/Qwen2.5-VL-3B-Instruct",
    clip_model="openai/clip-vit-large-patch14",
    seed=42, seeds=[42, 43, 44],
    n_train=2000, n_test=500, conflict_frac=0.5, max_img_side=448,
    num_generations=4, max_completion_len=256,
    lr=1e-5, max_steps=300, per_device_bs=4, grad_accum=2, beta=0.04,
    lora_r=16, four_bit_policy=False, loss_type="grpo", gspo=False,
    lora_targets="attn_mlp",      # "attn" | "attn_mlp" | "attn_mlp_merger" (also adapts the vision->language projector)
    use_dora=False,               # DoRA (weight-decomposed LoRA) variant
    eval_four_bit=False,          # evaluate in bf16, same precision as training (4-bit eval of a bf16-trained adapter loses accuracy)
    use_vllm=False,               # True = much faster generation in training (needs: pip install vllm; A100/L4)
    attn_n=50,                    # items for the layer-wise visual-attention analysis
    bootstrap_n=5000, pope_n=500, cov_risk_n=200, cov_risk_k=5, blank_n=200,
    # Reward A hyper-parameters (justify these in the paper; ablated in Section 11)
    A_w_consistency=0.5, A_w_grounding=0.5, A_clip_topk=10, A_min_think_words=15,
    A_use_q=True, A_asked_check=True, A_gate_mode="starved",   # switches used by the ablation notebook ("starved" | "always" | "inverse")
    # PAPO (Wang et al., ICLR 2026): maximise KL(pi(.|q,I) || pi(.|q,I_mask)) + double entropy. Check gamma/eta in its Appendix C.
    papo_gamma=0.01, papo_eta1=0.03, papo_eta2=0.03, papo_mask_ratio=0.6,
    # CFPO (arXiv 2606.23206): maximise KL to a counterfactual with high-saliency visual information replaced by its mean
    cfpo_gamma=0.02, cfpo_lambda=2.0,
    urdu_n=300,                                           # idea F: ~300 translated items
    judge_model="Qwen/Qwen2.5-VL-7B-Instruct",          # separate, larger judge (4-bit, ~6 GB)
    second_model="HuggingFaceTB/SmolVLM2-2.2B-Instruct",   # second model family for G4
)
PRESETS = {
    "smoke": dict(n_train=48, n_test=24, max_steps=6, seeds=[42], pope_n=24, cov_risk_n=12, cov_risk_k=3, blank_n=12, bootstrap_n=500),
    "pilot": dict(n_train=400, n_test=100, max_steps=50, seeds=[42], pope_n=100, cov_risk_n=50, blank_n=50),
    "main":  dict(),
}
CFG.update(PRESETS[CFG["preset"]])
# SAME BUDGET AS MAIN PART 1; seed 42 only (part 1 models)
CFG.update(max_steps=40, lr=2e-5, n_test=200, pope_n=200, cov_risk_n=60, cov_risk_k=4, blank_n=60, seeds=[42])

# Automatic settings for small GPUs (T4 / P100 / V100: 16 GB, no bf16). Remove this block to control everything by hand.
import torch
if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / 2**30
    if gb < 30 or not torch.cuda.is_bf16_supported():
        CFG.update(four_bit_policy=True, per_device_bs=2, grad_accum=4, max_completion_len=min(CFG["max_completion_len"], 192))
        print(f"small GPU ({torch.cuda.get_device_name(0)}, {gb:.0f} GB): 4-bit policy, batch 2 x 4, completion <= 192 tokens")
else:
    print("WARNING: no GPU found. Turn on a GPU accelerator (Kaggle: GPU T4 x2; Colab: Runtime > Change runtime type > GPU).")

if CFG["drive"]:
    from google.colab import drive; drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/A3_TrackC")
else:
    # Kaggle first: Kaggle images also contain a /content folder, but only /kaggle/working is saved as notebook output
    ROOT = Path("/kaggle/working/a3") if Path("/kaggle/working").exists() else Path("/content/a3") if Path("/content").exists() else Path("./a3")
RES = ROOT/"results"; FIG = RES/"figures"; TAB = RES/"tables"; RUNS = ROOT/"runs"
for d in (ROOT, RES, FIG, TAB, RUNS): d.mkdir(parents=True, exist_ok=True)
REGISTRY = RES/"registry.jsonl"          # one line per finished run (resume + tables)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except Exception: pass
set_seed(CFG["seed"])
GUARD_KEYS = ["model", "max_steps", "n_train", "n_test", "lr", "num_generations", "per_device_bs", "grad_accum",
              "max_completion_len", "conflict_frac", "seed", "beta", "lora_r", "lora_targets", "four_bit_policy"]
_old = RES/"config.json"
if _old.exists():
    try:
        prev = json.load(open(_old))
        diff = {k: (prev.get(k), CFG.get(k)) for k in GUARD_KEYS if prev.get(k) != CFG.get(k)}
    except Exception:
        diff = {"unreadable": True}
    if diff:   # earlier runs used other settings -> move them aside instead of silently reusing them
        tag = time.strftime("%Y%m%d_%H%M%S")
        for dname in ("runs", "results"):
            if (ROOT/dname).exists(): shutil.move(str(ROOT/dname), str(ROOT/f"{dname}_old_{tag}"))
        for d in (RES, FIG, TAB, RUNS): d.mkdir(parents=True, exist_ok=True)
        print("Settings changed since the last run", diff, "-> old runs/results moved to *_old_" + tag)
json.dump(CFG, open(RES/"config.json", "w"), indent=1)
print("ROOT =", ROOT); print(CFG)

## 2. Data: modality-conflict benchmark from COCO val2017
* **Conflict item:** the question asks about an object that is *not* in the image → gold answer `None`.
* **Normal item:** the object is present → gold answer = its count (capped at 9).
* Three question phrasings are used in training; a **fourth, unseen phrasing** is used for the test set, so a model cannot win by memorising one template.
* COCO annotations are used **only** to create gold labels and for evaluation metrics. No reward reads them (this keeps the "annotation-free" claim honest).

In [ ]:
def download(url, dst):
    """Download with a visible progress line; resumes cleanly if a previous attempt left a partial file."""
    dst = Path(dst); tmp = dst.with_suffix(dst.suffix + ".part")
    if dst.exists(): return dst
    print("downloading", url, flush=True); t0 = time.time(); last = [0]
    def hook(blocks, bsize, total):
        done = blocks * bsize
        if done - last[0] >= 50 * 2**20 or (total > 0 and done >= total):
            last[0] = done; mb = done / 2**20; tot = total / 2**20 if total > 0 else float("nan")
            print(f"  {mb:,.0f} / {tot:,.0f} MB  ({mb / max(1e-6, time.time() - t0):.1f} MB/s)", flush=True)
    urllib.request.urlretrieve(url, tmp, reporthook=hook); tmp.rename(dst)
    return dst

def load_coco():
    # on Colab+Drive keep the 5,000 COCO images on the fast local disk; runs/results still go to Drive
    d = (Path("/content/a3data") if CFG["drive"] else ROOT)/"coco"; d.mkdir(parents=True, exist_ok=True)
    if not (d/"val2017").exists():
        z = download("http://images.cocodataset.org/zips/val2017.zip", d/"val2017.zip")
        print("extracting images ...", flush=True); zipfile.ZipFile(z).extractall(d); os.remove(z); print("images ready", flush=True)
    ann = d/"annotations"/"instances_val2017.json"
    if not ann.exists():
        z = download("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", d/"ann.zip")
        zipfile.ZipFile(z).extract("annotations/instances_val2017.json", d); os.remove(z)
    return json.load(open(ann)), d/"val2017"

TRAIN_TMPLS = ["How many {cats} are in this image? If there is none, answer None.",
               "Count the {cats} you can see in the picture. Answer None if there are no {cats}.",
               "What is the number of {cats} in the image? Reply None if the image has no {cat}."]
TEST_TMPL   =  "Tell me how many {cats} appear in this photo. If no {cat} appears, the answer is None."
IRREG = {"person": "people", "sheep": "sheep", "skis": "skis", "knife": "knives", "bus": "buses", "wine glass": "wine glasses",
         "mouse": "computer mice", "couch": "couches", "sandwich": "sandwiches", "toothbrush": "toothbrushes", "bench": "benches",
         "scissors": "scissors", "hot dog": "hot dogs", "broccoli": "broccoli", "giraffe": "giraffes"}
def plural(c): return IRREG.get(c, c + "s")

def build_items(coco, img_dir, n, conflict_frac, seed, tmpls, name_map=None, exclude_ids=()):
    rng = random.Random(seed)
    cats = {c["id"]: c["name"] for c in coco["categories"]}
    per_img = collections.defaultdict(collections.Counter)
    for a in coco["annotations"]:
        if not a.get("iscrowd", 0): per_img[a["image_id"]][cats[a["category_id"]]] += 1
    files = {im["id"]: im["file_name"] for im in coco["images"]}
    ids = sorted(i for i in per_img if per_img[i] and i not in set(exclude_ids)); rng.shuffle(ids)
    allc = sorted(cats.values()); out = []
    for i in ids:
        if len(out) >= n: break
        present = per_img[i]
        if rng.random() < conflict_frac:
            asked = rng.choice([c for c in allc if c not in present]); sol, kind = "None", "conflict"
        else:
            asked = rng.choice(sorted(present)); sol, kind = str(min(present[asked], 9)), "normal"
        disp = (name_map or {}).get(asked, asked)
        out.append(dict(image=str(Path(img_dir)/files[i]), question=rng.choice(tmpls).format(cat=disp, cats=plural(disp) if not name_map else disp), solution=sol,
                        kind=kind, present=sorted(present), counts=dict(present), asked=asked, img_id=i))
    return out

## 3. Prompt, parsing and shared text utilities

In [ ]:
SYSTEM = ("You are a careful visual assistant. First describe the visual evidence step by step inside <think></think>, "
          "mentioning which objects you actually see. Then give only the final answer (a number, or None if the asked "
          "object is not in the image) inside <answer></answer>.")
def make_prompt(question, system=SYSTEM):
    return [{"role": "system", "content": [{"type": "text", "text": system}]},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]

_TH = re.compile(r"<think>(.*?)</think>", re.S); _AN = re.compile(r"<answer>(.*?)</answer>", re.S)
def ctext(c): return c[0]["content"] if isinstance(c, list) else c
def parse(text):
    t = _TH.search(text or ""); a = _AN.search(text or "")
    return (t.group(1).strip() if t else None), (a.group(1).strip() if a else None)

URDU_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
NUM_WORDS = {"zero": 0, "one": 1, "a single": 1, "single": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
             "eight": 8, "nine": 9, "ten": 10, "a couple of": 2, "a pair of": 2}
NONE_WORDS = {"none", "no", "0", "zero", "nothing", "there is none", "not present", "n/a", "کوئی نہیں", "کچھ نہیں", "نہیں", "صفر"}
def norm_ans(a):
    if a is None: return None
    a = a.translate(URDU_DIGITS).strip().lower().strip(" .۔!")
    if a in NONE_WORDS or a.startswith("none"): return "none"
    m = re.search(r"\d+", a)
    if m: return "none" if int(m.group(0)) == 0 else m.group(0)
    for w, v in NUM_WORDS.items():
        if re.search(r"\b%s\b" % w, a): return str(v) if v else "none"
    return a
def is_correct(text, solution): return norm_ans(parse(text)[1]) == norm_ans(solution)

COCO80 = ["person","bicycle","car","motorcycle","airplane","bus","train","truck","boat","traffic light","fire hydrant","stop sign",
 "parking meter","bench","bird","cat","dog","horse","sheep","cow","elephant","bear","zebra","giraffe","backpack","umbrella","handbag",
 "tie","suitcase","frisbee","skis","snowboard","sports ball","kite","baseball bat","baseball glove","skateboard","surfboard",
 "tennis racket","bottle","wine glass","cup","fork","knife","spoon","bowl","banana","apple","sandwich","orange","broccoli","carrot",
 "hot dog","pizza","donut","cake","chair","couch","potted plant","bed","dining table","toilet","tv","laptop","mouse","remote",
 "keyboard","cell phone","microwave","oven","toaster","sink","refrigerator","book","clock","vase","scissors","teddy bear",
 "hair drier","toothbrush"]
SYN = {"person": ["person", "people", "man", "men", "woman", "women", "child", "children", "boy", "girl", "kid", "pedestrian", "player", "persons"],
       "tv": ["tv", "television", "monitor"], "couch": ["couch", "sofa"], "motorcycle": ["motorcycle", "motorbike"],
       "bicycle": ["bicycle", "bike"], "airplane": ["airplane", "plane", "aeroplane", "aircraft"], "dining table": ["dining table", "table"],
       "cell phone": ["cell phone", "phone", "smartphone", "mobile"], "potted plant": ["potted plant", "plant"], "sports ball": ["ball", "sports ball"],
       "knife": ["knife", "knives"], "donut": ["donut", "doughnut"], "hair drier": ["hair drier", "hair dryer"], "refrigerator": ["refrigerator", "fridge"],
       "mouse": ["computer mouse", "mouse", "mice"], "remote": ["remote", "remote control"], "car": ["car", "cars"], "sheep": ["sheep"], "skis": ["skis", "ski"]}
NEG = re.compile(r"\b(no|not|none|without|cannot|can't|isn't|aren't|absent|nor|neither|lack|lacks|missing|nothing|never|don't|doesn't)\b")
def _pat(w): return re.compile(r"\b%s(?:s|es)?\b" % re.escape(w))
VOCAB = {c: [_pat(w) for w in SYN.get(c, [c])] for c in COCO80}
def sentences(t): return [s.strip() for s in re.split(r"[.!?;\n]", (t or "").lower()) if s.strip()]
def objects_in(s): return {c for c, ps in VOCAB.items() if any(p.search(s) for p in ps)}
def affirmed_objects(think):
    """Objects named in non-negated sentences (the model claims to SEE them)."""
    return set().union(*[objects_in(s) for s in sentences(think) if not NEG.search(s)]) if think else set()
def denied_objects(think):
    return set().union(*[objects_in(s) for s in sentences(think) if NEG.search(s)]) if think else set()
def asked_object(question):
    q = question.lower(); hits = [c for c in COCO80 if any(p.search(q) for p in VOCAB[c])]
    return max(hits, key=len) if hits else None
def numbers_in(s):
    s = s.translate(URDU_DIGITS); out = [int(x) for x in re.findall(r"\b\d+\b", s)]
    out += [v for w, v in NUM_WORDS.items() if re.search(r"\b%s\b" % w, s)]
    return out

## 3b. Basic rewards (format, outcome)

In [ ]:
def r_format(completions, **kw):
    out = []
    for c in completions:
        th, an = parse(ctext(c)); out.append(1.0 if (th is not None and an is not None and len(th.split()) >= 5) else 0.0)
    return out

def r_outcome(completions, solution, **kw):
    """Binary verifiable outcome reward (exact match after normalisation, incl. None)."""
    return [1.0 if is_correct(ctext(c), s) else 0.0 for c, s in zip(completions, solution)]

## 4. Reward A — judge-free, annotation-free consistency reward (core novelty)
A completion gets reward

$$R_A = Q \cdot \big(w_c\,C + w_g\,G\big),\qquad w_c + w_g = 1$$

* **C — internal consistency (text only).** Does the reasoning agree with the final answer about the *asked* object (parsed from the question)?
  - answer `None` → the reasoning must not claim to see the object (C=0 if it does), and is rewarded for explicitly ruling it out (C=1), otherwise C=0.6.
  - answer = k → the reasoning must claim to see the object (else C=0.2); if it states a number for that object, it must equal k (C=1 vs 0.3); no number → C=0.7.
* **G — visual self-verification (no annotations, no judge).** A frozen CLIP model checks the objects the reasoning *claims to see* against the image: for each affirmed object, it is "verified" when its zero-shot rank among the 80 class names is in the top-K. G = verified / claimed. The asked object is also checked: answering `None` while CLIP ranks the object in the top-3 (or answering a count while CLIP ranks it below 30) costs 0.5. CLIP is pre-trained, frozen, and never sees COCO labels.
* **Q — anti-hacking quality gate.** Q = length factor × (1 − question-copy penalty) × bigram diversity. Stops empty, copied or repetitive reasoning from collecting reward.

Nothing in $R_A$ reads `solution`, `present` or any annotation. Differences from nearest works: PAPO changes the *loss* with masked-image KL; CORA/PEARL use learned critics or perception tasks; Faithful-R1/DeFacto use annotations or a judge. A is a reward-level, model-free check of the written reasoning.

In [ ]:
import torch
from PIL import Image
HAVE_GPU = torch.cuda.is_available()
DEV = "cuda" if HAVE_GPU else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if HAVE_GPU else None)

def load_img(path, side=None):
    im = Image.open(path).convert("RGB"); side = side or CFG["max_img_side"]
    if max(im.size) > side:
        r = side / max(im.size); im = im.resize((max(28, int(im.width*r)), max(28, int(im.height*r))))
    return im

# ---------- C: internal consistency ----------
def consistency_score(think, answer, question):
    a = norm_ans(answer); obj = asked_object(question)
    if a is None or think is None: return 0.0
    if obj is None: return 0.5
    aff, den = affirmed_objects(think), denied_objects(think)
    if a == "none":
        if obj in aff: return 0.0
        return 1.0 if obj in den else 0.6
    if not a.isdigit(): return 0.0
    if obj not in aff: return 0.2
    nums = [n for s in sentences(think) if obj in objects_in(s) and not NEG.search(s) for n in numbers_in(s)]
    if not nums: return 0.7
    return 1.0 if int(a) in nums else 0.3

# ---------- Q: anti-hacking quality gate ----------
def quality_gate(think, question):
    if not think: return 0.0
    w = think.lower().split()
    length = min(1.0, len(w) / CFG["A_min_think_words"])
    q = set(question.lower().split()); copy = np.mean([x in q for x in w])
    copy_pen = 1.0 - max(0.0, copy - 0.5) * 2
    bg = list(zip(w, w[1:])); div = (len(set(bg)) / len(bg)) if bg else 0.0
    return float(max(0.0, length * copy_pen * min(1.0, div / 0.6)))

# ---------- G: CLIP self-verification ----------
# CLIP features computed from the sub-modules: works on old and new transformers
# (from v5 `get_text_features` returns an output object instead of a tensor)
def clip_text(model, input_ids=None, attention_mask=None, **kw):
    return model.text_projection(model.text_model(input_ids=input_ids, attention_mask=attention_mask).pooler_output)
def clip_image(model, pixel_values=None, **kw):
    return model.visual_projection(model.vision_model(pixel_values=pixel_values).pooler_output)

class ClipVerifier:
    def __init__(self, name=None):
        from transformers import CLIPModel, CLIPProcessor
        name = name or CFG["clip_model"]
        self.m = CLIPModel.from_pretrained(name, **{DTYPE_KEY: torch.float16 if HAVE_GPU else torch.float32}).to(DEV).eval()
        self.p = CLIPProcessor.from_pretrained(name); self.cache = {}
        prompts = [[f"a photo of a {c}.", f"a photo containing a {c}.", f"a close-up photo of a {c}."] for c in COCO80]
        with torch.no_grad():
            t = self.p(text=sum(prompts, []), return_tensors="pt", padding=True).to(DEV)
            e = clip_text(self.m, **t); e = e / e.norm(dim=-1, keepdim=True)
            self.text = e.view(len(COCO80), 3, -1).mean(1); self.text = self.text / self.text.norm(dim=-1, keepdim=True)
    @torch.no_grad()
    def ranks(self, path):
        """rank (0 = best) of each of the 80 class names for this image."""
        if path not in self.cache:
            x = self.p(images=load_img(path, 336), return_tensors="pt").to(DEV)
            v = clip_image(self.m, pixel_values=x["pixel_values"].to(self.m.dtype))
            v = v / v.norm(dim=-1, keepdim=True); s = (v @ self.text.T)[0].float().cpu().numpy()
            order = np.argsort(-s); r = np.empty_like(order); r[order] = np.arange(len(order))
            self.cache[path] = dict(zip(COCO80, r.tolist()))
        return self.cache[path]
CLIPV = None
def get_clip():
    global CLIPV
    if CLIPV is None: CLIPV = ClipVerifier()
    return CLIPV

def grounding_score(think, answer, question, image_path):
    if think is None: return 0.0
    rk = get_clip().ranks(image_path); K = CFG["A_clip_topk"]
    claimed = affirmed_objects(think)
    g = (np.mean([rk[o] < K for o in claimed]) if claimed else 0.5)
    obj, a = asked_object(question), norm_ans(answer)
    if CFG["A_asked_check"] and obj is not None and a is not None:
        if a == "none" and rk[obj] < 3: g -= 0.5
        if a.isdigit() and rk[obj] >= 30: g -= 0.5
    return float(max(0.0, g))

def _A(completions, question, image_path, wc, wg, use_q=None):
    use_q = CFG["A_use_q"] if use_q is None else use_q; out = []
    if wc + wg <= 0: return [float(quality_gate(parse(ctext(c))[0], q)) if use_q else 0.0 for c, q in zip(completions, question)]
    for c, q, ip in zip(completions, question, image_path):
        th, an = parse(ctext(c))
        if th is None or an is None: out.append(0.0); continue
        C = consistency_score(th, an, q); G = grounding_score(th, an, q, ip) if wg > 0 else 0.0
        Q = quality_gate(th, q) if use_q else 1.0
        out.append(float(Q * (wc * C + wg * G) / (wc + wg)))
    return out

# Reward A and its ablations (all annotation-free and judge-free; they never read `solution`)
def r_A(completions, question, image_path, **kw):        return _A(completions, question, image_path, CFG["A_w_consistency"], CFG["A_w_grounding"])
def r_A_text(completions, question, image_path, **kw):   return _A(completions, question, image_path, 1.0, 0.0)          # C·Q only
def r_A_clip(completions, question, image_path, **kw):   return _A(completions, question, image_path, 0.0, 1.0)          # G·Q only
def r_A_noQ(completions, question, image_path, **kw):    return _A(completions, question, image_path, CFG["A_w_consistency"], CFG["A_w_grounding"], use_q=False)

# ---------- A-gated: starvation-gated consistency reward (novelty linking A and D) ----------
GATE = dict(G=None, log=[])          # G is set by train() to the run's num_generations
def r_A_gated(completions, question, image_path, solution, **kw):
    """Reward A is given ONLY inside groups whose outcome reward has zero variance (all right or all wrong),
    i.e. exactly where GRPO would get no gradient. In groups with a learning signal from the outcome reward it is 0,
    so A cannot dominate or be exploited there. Uses `solution` only to compute the outcome gate, never inside A."""
    G = GATE["G"] or CFG["num_generations"]
    A = np.array(_A(completions, question, image_path, CFG["A_w_consistency"], CFG["A_w_grounding"]))
    if len(A) % G:
        GATE["log"].append(dict(gated_share=None)); return A.tolist()      # cannot form groups -> ungated fallback
    o = np.array(r_outcome(completions, solution)).reshape(-1, G)
    starved = o.std(1) < 1e-8
    gate = np.repeat({"starved": starved, "inverse": ~starved, "always": np.ones_like(starved)}[CFG["A_gate_mode"]], G).astype(float)
    GATE["log"].append(dict(gated_share=float(gate.mean()), mean_A_in_gate=float(A[gate > 0].mean()) if gate.any() else None))
    return (A * gate).tolist()

### 4b. Unit tests for reward A (hand-made hacking cases)

In [ ]:
def _c(t): return [{"content": t}]
Q_ = "How many cats are in this image? If there is none, answer None."
CASES = {
 "good_none":     "<think>I see a man sitting on a bench next to a dog. There is no cat anywhere in the picture.</think><answer>None</answer>",
 "good_count":    "<think>Two cats are lying on the bed, and a remote is next to them on the blanket in the room.</think><answer>2</answer>",
 "contradiction": "<think>I can see a cat sleeping on the sofa in the living room, near the window light.</think><answer>None</answer>",
 "wrong_number":  "<think>Three cats are lying on the bed, and a remote is next to them on the blanket.</think><answer>2</answer>",
 "empty_think":   "<think>ok</think><answer>None</answer>",
 "copy_question": "<think>How many cats are in this image? If there is none, answer None.</think><answer>None</answer>",
 "repetitive":    "<think>cat cat cat cat cat cat cat cat cat cat cat cat cat cat cat cat</think><answer>1</answer>",
 "no_format":     "None",
}
rows = []
for k, t in CASES.items():
    th, an = parse(t)
    rows.append(dict(case=k, C=consistency_score(th, an, Q_) if th else 0, Q=round(quality_gate(th, Q_), 2) if th else 0,
                     A_text=r_A_text([_c(t)], [Q_], ["-"])[0]))
UT = pd.DataFrame(rows); print(UT.to_string(index=False))
assert UT.set_index("case").A_text["good_none"] > 0.8 and UT.set_index("case").A_text["good_count"] > 0.8
for bad in ["contradiction", "empty_think", "copy_question", "repetitive", "no_format"]:
    assert UT.set_index("case").A_text[bad] < 0.35, bad
UT.to_csv(RES/"rewardA_unit_tests.csv", index=False); print("reward A unit tests passed")

# A-gated test: group 1 = all correct (starved -> A applied), group 2 = mixed outcome (A switched off)
_saveG, _saveW = GATE["G"], CFG["A_w_grounding"]; GATE["G"] = 2; CFG["A_w_grounding"] = 0.0
_g = r_A_gated([_c(CASES["good_none"]), _c(CASES["copy_question"]), _c(CASES["good_none"]), _c("<think>a b</think><answer>3</answer>")],
               [Q_] * 4, ["-"] * 4, ["None", "None", "None", "None"])
print("A-gated:", [round(x, 2) for x in _g]); assert _g[0] > 0.8 and _g[1] < 0.35 and _g[2] == 0 and _g[3] == 0
GATE["G"], CFG["A_w_grounding"] = _saveG, _saveW; print("A-gated test passed")

## 5. Other reward families (for the head-to-head, idea B)
| Family | Function | Needs | Notes |
|---|---|---|---|
| MC-R1 (base paper, re-implemented) | `r_mcr1` | gold answer | three string-match rewards from the MC-R1 abstract: irrelevant-information-excluded **weighted ROUGE-L** (dependency-parse token weights), **Key Token Overlap Rate**, **Reasoning Faithfulness Constraint**; `None` = gold for conflict items, penalised on non-conflict items. **Weights in `MCR1` are our reconstruction — check them against the equations in the full paper.** |
| Dense outcome | `r_dense` | gold answer | partial credit by count distance; breaks reward ties → fewer zero-variance groups (idea D) |
| PAPO-style visual dependence | `r_papo` | frozen VLM | reward-level approximation of PAPO: σ(s·(log p(y∣real img) − log p(y∣grey img))). PAPO itself is a loss term; state this clearly in the paper |
| VLM judge | `r_judge` | frozen VLM judge | P(Yes) that the reasoning describes the image correctly (judge-based family) |
| PAPO (loss) | `papo_loss` | masked image | implicit perception loss + double entropy (Section 8b) |
| CFPO-style (loss) | `cfpo` | counterfactual visual tokens | counterfactual KL (Section 8b) |
| Separate 7B judge | `r_judge7b` | Qwen2.5-VL-7B (4-bit) | judge with different weights than the policy |
| A (ours) | `r_A` | CLIP + text | judge-free, annotation-free |

In [ ]:
# ---------- MC-R1 (Chen et al., IEEE TMM 2026) — re-implemented from the paper's description ----------
MCR1 = dict(w_rouge=0.4, w_ktor=0.3, w_rfc=0.3,        # weights of the three string-match rewards
            none_correct=1.0, none_wrong_pen=-0.5,       # gold None: reward for answering None; non-conflict item answered None: penalty
            format_w=0.2)
DEP_W = {"ROOT": 1.0, "nsubj": 1.0, "nsubjpass": 1.0, "dobj": 1.0, "attr": 1.0, "nummod": 1.0, "neg": 1.0,
         "pobj": 0.8, "amod": 0.8, "compound": 0.8, "acomp": 0.8, "conj": 0.8, "appos": 0.8, "advmod": 0.5}
EXCL_POS = {"DET", "PUNCT", "AUX", "ADP", "CCONJ", "SCONJ", "PART", "SPACE", "PRON", "INTJ", "SYM"}   # irrelevant information -> weight 0
KEY_POS = {"NOUN", "PROPN", "NUM", "ADJ", "VERB"}
_STOP = set("a an the is are was were be been of in on at to for with and or but this that these those it its there here i you he she we they "
            "my your his her our their as by from into than then so very can could would should will do does did has have had not".split())
_NLP = None
def nlp():
    global _NLP
    if _NLP is None:
        try:
            import spacy; _NLP = spacy.load("en_core_web_sm", disable=["ner"])
        except Exception as e:
            print("spaCy unavailable -> heuristic weights (", e, ")"); _NLP = False
    return _NLP
def weighted_tokens(text):
    """[(lemma, weight, is_key)] — dependency-weighted tokens; function words get weight 0 (excluded)."""
    text = (text or "").strip()
    if not text: return []
    P = nlp()
    if P:
        out = []
        for t in P(text):
            if t.pos_ in EXCL_POS and t.dep_ != "neg": continue
            out.append((t.lemma_.lower(), DEP_W.get(t.dep_, 0.5), t.pos_ in KEY_POS or t.like_num))
        return out
    toks = re.findall(r"[a-z0-9]+", text.lower())
    return [(w, 1.0 if w.isdigit() else 0.8, True) for w in toks if w not in _STOP]
def weighted_rouge_l(pred, ref):
    """IIE weighted ROUGE-L F1: weighted LCS over content tokens (weights of the reference tokens)."""
    a, b = weighted_tokens(pred), weighted_tokens(ref)
    if not a or not b: return float(norm_ans(pred) == norm_ans(ref)) if (pred and ref) else 0.0
    n, m = len(a), len(b); D = np.zeros((n + 1, m + 1))
    for i in range(n):
        for j in range(m):
            D[i+1, j+1] = D[i, j] + b[j][1] if a[i][0] == b[j][0] else max(D[i, j+1], D[i+1, j])
    lcs = D[n, m]; P_ = lcs / max(1e-9, sum(w for _, w, _ in a)); R_ = lcs / max(1e-9, sum(w for _, w, _ in b))
    return float(0.0 if P_ + R_ == 0 else 2 * P_ * R_ / (P_ + R_))
def key_tokens(text): return {w for w, _, k in weighted_tokens(text) if k}
def ktor(pred, ref):
    """Key Token Overlap Rate = |K(pred) ∩ K(ref)| / |K(ref)|."""
    kr = key_tokens(ref)
    return float(len(key_tokens(pred) & kr) / len(kr)) if kr else float(norm_ans(pred) == norm_ans(ref))
def reasoning_faithfulness(think, answer):
    """RFC: the final answer must be supported by the reasoning. Key tokens of the answer must appear in <think>;
    a None answer must be preceded by an explicit absence statement in <think>."""
    if not think or answer is None: return 0.0
    if norm_ans(answer) == "none": return 1.0 if NEG.search(think.lower()) else 0.0
    ka = key_tokens(answer) | ({norm_ans(answer)} if (norm_ans(answer) or "").isdigit() else set())
    kt = key_tokens(think) | {str(n) for s_ in sentences(think) for n in numbers_in(s_)}
    return float(len(ka & kt) / len(ka)) if ka else 0.0
def _canon(a):
    n = norm_ans(a); return n if (n is not None and (n == "none" or n.isdigit())) else (a or "")
def mcr1_score(text, solution):
    th, an = parse(text); fmt = MCR1["format_w"] * float(th is not None and an is not None)
    if an is None: return fmt
    a, g = _canon(an), _canon(solution)
    if g == "none": return fmt + (MCR1["none_correct"] if a == "none" else 0.0) * (0.5 + 0.5 * reasoning_faithfulness(th, an))
    if a == "none": return fmt + MCR1["none_wrong_pen"]
    return fmt + MCR1["w_rouge"] * weighted_rouge_l(a, g) + MCR1["w_ktor"] * ktor(a, g) + MCR1["w_rfc"] * reasoning_faithfulness(th, an)
def r_mcr1(completions, solution, **kw):
    return [mcr1_score(ctext(c), s) for c, s in zip(completions, solution)]

def r_dense(completions, solution, **kw):
    out = []
    for c, s in zip(completions, solution):
        a, g = norm_ans(parse(ctext(c))[1]), norm_ans(s)
        if a is None: out.append(0.0)
        elif a == g: out.append(1.0)
        elif g == "none" or a == "none" or not a.isdigit(): out.append(0.0)
        else: out.append(float(max(0.0, 1 - abs(int(a) - int(g)) / max(1, int(g)))) * 0.7)
    return out

# ---------- frozen reference VLM (shared by r_papo, r_judge and the image-dependence metric) ----------
from transformers import AutoProcessor, BitsAndBytesConfig
from transformers import AutoModelForImageTextToText
try:
    from transformers import Qwen2_5_VLForConditionalGeneration
except ImportError:
    Qwen2_5_VLForConditionalGeneration = None
def is_qwen(name): return "qwen" in name.lower()
def VLM_for(name): return Qwen2_5_VLForConditionalGeneration if (is_qwen(name) and Qwen2_5_VLForConditionalGeneration) else AutoModelForImageTextToText
DT = torch.bfloat16 if (HAVE_GPU and torch.cuda.is_bf16_supported()) else torch.float16
# transformers >= 4.56 uses `dtype`; older versions use `torch_dtype` (an ignored key silently loads bf16 weights)
import transformers as _tf
from packaging import version as _v
DTYPE_KEY = "dtype" if _v.parse(_tf.__version__) >= _v.parse("4.56") else "torch_dtype"
def bnb4(): return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=DT, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
def get_processor(name=None):
    name = name or CFG["model"]
    if is_qwen(name):
        p = AutoProcessor.from_pretrained(name, min_pixels=64*28*28, max_pixels=320*28*28, padding_side="left")
    else:   # SmolVLM / Idefics3: one 384px tile per image, no splitting (keeps token count comparable)
        p = AutoProcessor.from_pretrained(name, padding_side="left")
        try: p.image_processor.do_image_splitting = False; p.image_processor.size = {"longest_edge": 384}
        except Exception: pass
    p._nested_images = not is_qwen(name)
    if p.tokenizer.pad_token is None: p.tokenizer.pad_token = p.tokenizer.eos_token
    return p
def proc_call(proc, texts, images, **kw):
    """Qwen expects a flat image list; Idefics3/SmolVLM expects one list per sample."""
    imgs = [[im] for im in images] if getattr(proc, "_nested_images", False) else images
    return proc(text=texts, images=imgs, return_tensors="pt", padding=True, **kw)
REF = None
def get_ref():
    global REF
    if REF is None:
        m = VLM_for(CFG["model"]).from_pretrained(CFG["model"], **{DTYPE_KEY: DT}, quantization_config=bnb4(), device_map="auto").eval()
        REF = (m, get_processor())
    return REF
def grey_like(im): return Image.new("RGB", im.size, (128, 128, 128))

@torch.no_grad()
def seq_logprob(model, proc, question, image, completion):
    """Mean log-prob of `completion` tokens given (question, image)."""
    msgs = make_prompt(question)
    p_txt = proc.apply_chat_template(msgs, add_generation_prompt=True)
    full = p_txt + completion
    a = proc(text=[p_txt], images=[image], return_tensors="pt"); b = proc(text=[full], images=[image], return_tensors="pt").to(model.device)
    n0 = a["input_ids"].shape[1]
    logits = model(**b).logits[0, :-1].float(); tgt = b["input_ids"][0, 1:]
    lp = torch.log_softmax(logits, -1).gather(1, tgt[:, None])[:, 0][n0-1:]
    return float(lp.mean()) if lp.numel() else 0.0

def r_papo(completions, question, image_path, **kw):
    m, p = get_ref(); out = []
    for c, q, ip in zip(completions, question, image_path):
        im = load_img(ip); t = ctext(c)
        d = seq_logprob(m, p, q, im, t) - seq_logprob(m, p, q, grey_like(im), t)
        out.append(float(1 / (1 + math.exp(-8 * d))))
    return out

JUDGE_Q = ("Here is a description written about this image:\n\"{d}\"\nDoes the description only mention objects that are really "
           "in the image, and is its final conclusion correct for the image? Answer Yes or No.")
JUDGE = None
def get_judge():
    """Separate, larger judge (default Qwen2.5-VL-7B, 4-bit). Different weights from the policy -> less self-preference bias."""
    global JUDGE
    if JUDGE is None:
        n = CFG["judge_model"]; m = VLM_for(n).from_pretrained(n, **{DTYPE_KEY: DT}, quantization_config=bnb4(), device_map="auto").eval()
        JUDGE = (m, get_processor(n))
    return JUDGE

@torch.no_grad()
def _judge(model_proc, completions, question, image_path):
    m, p = model_proc; out = []
    yes = p.tokenizer.encode("Yes", add_special_tokens=False)[0]; no = p.tokenizer.encode("No", add_special_tokens=False)[0]
    for c, q, ip in zip(completions, question, image_path):
        th, an = parse(ctext(c))
        if th is None or an is None: out.append(0.0); continue
        msg = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": JUDGE_Q.format(d=f"{th} Final answer to '{q}': {an}")[:1500]}]}]
        x = proc_call(p, [p.apply_chat_template(msg, add_generation_prompt=True)], [load_img(ip)]).to(m.device)
        lg = m(**x).logits[0, -1].float(); out.append(float(torch.softmax(lg[[yes, no]], 0)[0]))
    return out
def r_judge(completions, question, image_path, **kw):    return _judge(get_ref(), completions, question, image_path)    # same-size (3B) judge
def r_judge7b(completions, question, image_path, **kw):  return _judge(get_judge(), completions, question, image_path)  # separate 7B judge

# name -> (reward functions, weights). r_format is always included so every family gets the same format signal.
REWARD_SETS = {
    "outcome":      ([r_format, r_outcome], [0.2, 1.0]),
    "mcr1":         ([r_mcr1], [1.0]),
    "dense":        ([r_format, r_dense], [0.2, 1.0]),
    "papo_style":   ([r_format, r_outcome, r_papo], [0.2, 1.0, 0.5]),
    "judge":        ([r_format, r_outcome, r_judge], [0.2, 1.0, 0.5]),
    "judge7b":      ([r_format, r_outcome, r_judge7b], [0.2, 1.0, 0.5]),
    "papo_loss":    ([r_format, r_outcome], [0.2, 1.0]),     # + PAPO loss term (TRAINER_MODE)
    "cfpo":         ([r_format, r_outcome], [0.2, 1.0]),     # + CFPO loss term (TRAINER_MODE)
    "A_only":       ([r_format, r_A], [0.2, 1.0]),
    "outcome_A":    ([r_format, r_outcome, r_A], [0.2, 1.0, 0.5]),
    # ablations of A (Section 11)
    "outcome_A_text": ([r_format, r_outcome, r_A_text], [0.2, 1.0, 0.5]),
    "outcome_A_clip": ([r_format, r_outcome, r_A_clip], [0.2, 1.0, 0.5]),
    "outcome_A_noQ":  ([r_format, r_outcome, r_A_noQ], [0.2, 1.0, 0.5]),
    "outcome_A_gated": ([r_format, r_outcome, r_A_gated], [0.2, 1.0, 0.5]),   # A only where the outcome group is starved
}
TRAINER_MODE = {"papo_loss": "papo", "cfpo": "cfpo"}      # families that change the loss, not the reward

## 6. Build datasets

In [ ]:
coco, IMG_DIR = load_coco()
ALL_TR = build_items(coco, IMG_DIR, CFG["n_train"], CFG["conflict_frac"], CFG["seed"], TRAIN_TMPLS)
TRAIN = ALL_TR
TEST = build_items(coco, IMG_DIR, CFG["n_test"], CFG["conflict_frac"], CFG["seed"] + 1, [TEST_TMPL], exclude_ids={x["img_id"] for x in TRAIN})
assert not ({x["img_id"] for x in TRAIN} & {x["img_id"] for x in TEST}), "image leakage"
json.dump(dict(train=TRAIN, test=TEST), open(RES/"splits.json", "w"))
print(len(TRAIN), len(TEST), collections.Counter(x["kind"] for x in TRAIN), collections.Counter(x["kind"] for x in TEST))
pd.DataFrame([dict(split="train", **collections.Counter(x["kind"] for x in TRAIN)), dict(split="test", **collections.Counter(x["kind"] for x in TEST))]).to_csv(RES/"dataset_stats.csv", index=False)

## 7. Generation (policy + LoRA adapter)

In [ ]:
try:   # never route LoRA through torchao (an old Kaggle torchao raises ImportError inside PEFT)
    import peft.tuners.lora.torchao as _pt; _pt.is_torchao_available = lambda: False
except Exception:
    pass

def load_policy(adapter=None, four_bit=None, model=None, attn=None):
    name = model or CFG["model"]; four_bit = CFG["eval_four_bit"] if four_bit is None else four_bit
    extra = dict(attn_implementation=attn) if attn else {}
    m = VLM_for(name).from_pretrained(name, **{DTYPE_KEY: DT}, quantization_config=bnb4() if four_bit else None, device_map="auto", **extra)
    if adapter:
        from peft import PeftModel; m = PeftModel.from_pretrained(m, adapter)
    return m.eval(), get_processor(name)

@torch.no_grad()
def generate(model, proc, items, batch=8, blank=False, sample=False, n=1, max_new=None, system=SYSTEM):
    outs = []; max_new = max_new or CFG["max_completion_len"]
    for i in range(0, len(items), batch):
        b = items[i:i+batch]
        imgs = [load_img(x["image"]) for x in b]
        if blank: imgs = [grey_like(im) for im in imgs]
        texts = [proc.apply_chat_template(make_prompt(x["question"], system), add_generation_prompt=True) for x in b]
        inp = proc_call(proc, texts, imgs).to(model.device)
        kw = dict(do_sample=True, temperature=0.8, top_p=0.95, num_return_sequences=n) if sample else dict(do_sample=False)
        g = model.generate(**inp, max_new_tokens=max_new, **kw)
        dec = proc.batch_decode(g[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)
        outs += [dec[j*n:(j+1)*n] for j in range(len(b))] if n > 1 else dec
    return outs

## 8. GRPO training (LoRA) — cost logging (G4) and gradient-starvation recorder (D)
`GRPOConfig` arguments are filtered against the installed TRL version, so the cell works across TRL releases. `loss_type` switches GRPO / DAPO / Dr.GRPO; `gspo=True` uses sequence-level importance sampling (GSPO).

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig
from transformers import TrainerCallback
import transformers
transformers.logging.set_verbosity_error()          # fewer log lines -> the notebook page stays responsive

# T4 / P100 / V100 (compute capability < 8): TRL's chunked log-prob kernel uses an fp32-output GEMM that only exists on
# Ampere+ GPUs ("gemm input type ... only supported for compute capability 8.0 or higher"). Switch it to TRL's fp32 fallback.
try:
    import trl.kernels.chunked_logprob as _cl
    if HAVE_GPU and torch.cuda.get_device_capability(0)[0] < 8 and hasattr(_cl, "_MM_OUT_DTYPE"):
        _cl._MM_OUT_DTYPE = False; print("older GPU: TRL chunked-logprob fp32 fallback enabled")
except ImportError:
    pass


class GroupStats:
    """Per reward call: share of groups whose reward has zero std (no GRPO signal), mean reward, length, None-rate."""
    def __init__(self, G): self.G, self.rows, self.step = G, [], 0
    def wrap(self, fn, last=False):
        def f(completions, **kw):
            r = fn(completions, **kw); texts = [ctext(c) for c in completions]
            if len(r) % self.G == 0:
                a = np.array(r, float).reshape(-1, self.G)
                self.rows.append(dict(step=self.step, reward=fn.__name__, mean=float(a.mean()),
                                      zero_std_share=float((a.std(1) < 1e-8).mean()),
                                      len=float(np.mean([len(t.split()) for t in texts])),
                                      none_rate=float(np.mean([norm_ans(parse(t)[1]) == "none" for t in texts]))))
            return r
        f.__name__ = fn.__name__; return f

class StepCB(TrainerCallback):
    def __init__(self, gs): self.gs = gs
    def on_step_end(self, args, state, control, **kw): self.gs.step = state.global_step

# LoRA placement. Regex strings are matched against full module names. The vision encoder itself stays frozen in all settings.
_ATTN = "q_proj|k_proj|v_proj|o_proj"; _MLP = "gate_proj|up_proj|down_proj"
LORA_TARGETS = {
    "attn":            ["q_proj", "k_proj", "v_proj", "o_proj"],
    "attn_mlp":        rf"^(?!.*visual)(?!.*vision).*\.({_ATTN}|{_MLP})$",
    "attn_mlp_merger": rf"(^(?!.*visual)(?!.*vision).*\.({_ATTN}|{_MLP})$)|(.*merger\.mlp\.(0|2)$)|(.*connector.*proj$)",
}
def make_cfg(**kw):
    fields = set(inspect.signature(GRPOConfig.__init__).parameters)
    dropped = [k for k in kw if k not in fields]
    if dropped: print("GRPOConfig: ignoring unsupported args for this TRL version:", dropped)
    return GRPOConfig(**{k: v for k, v in kw.items() if k in fields})

def to_dataset(items):
    return Dataset.from_list([dict(prompt=make_prompt(x["question"]), image=load_img(x["image"]), image_path=x["image"],
                                   question=x["question"], solution=x["solution"]) for x in items])

def run_name(reward, seed, **ov):
    extra = "_".join(f"{k}{str(v).split('/')[-1]}" for k, v in sorted(ov.items()))
    return f"{reward}_s{seed}" + (f"_{extra}" if extra else "")

def train(reward, seed, _name=None, _items=None, **ov):
    """ov: overrides, e.g. beta=0.1, loss_type='dapo', gspo=True, lora_r=32, four_bit_policy=True, wA=1.0"""
    name = _name or run_name(reward, seed, **ov); out = RUNS/name
    if (out/"adapter_model.safetensors").exists() and (out/"cost.json").exists():
        print("skip (exists)", name); return out
    set_seed(seed); c = {**CFG, **ov}
    fns, w = REWARD_SETS[reward]; w = list(w)
    if "wA" in ov: w = [ov["wA"] if f.__name__.startswith("r_A") else x for f, x in zip(fns, w)]
    gs = GroupStats(c["num_generations"]); fns_w = [gs.wrap(f) for f in fns]
    GATE["G"] = c["num_generations"]; GATE["log"] = []
    mk = {DTYPE_KEY: DT}
    if c["four_bit_policy"]: mk.update(quantization_config=bnb4(), device_map={"": 0})
    args = make_cfg(output_dir=str(out), learning_rate=c["lr"], max_steps=c["max_steps"], seed=seed,
                    per_device_train_batch_size=c["per_device_bs"], gradient_accumulation_steps=c["grad_accum"],
                    num_generations=c["num_generations"], max_completion_length=c["max_completion_len"], max_prompt_length=None,
                    beta=c["beta"], reward_weights=w, loss_type=c["loss_type"],
                    importance_sampling_level="sequence" if c["gspo"] else "token",
                    temperature=1.0, bf16=DT == torch.bfloat16, fp16=DT == torch.float16, gradient_checkpointing=True,
                    logging_steps=1, save_strategy="no", report_to="none", remove_unused_columns=False, disable_tqdm=True,
                    log_completions=False, model_init_kwargs=mk,
                    use_vllm=c["use_vllm"], vllm_mode="colocate", vllm_gpu_memory_utilization=0.35)
    peft = LoraConfig(r=c["lora_r"], lora_alpha=2*c["lora_r"], lora_dropout=0.05, use_dora=c["use_dora"],
                      target_modules=LORA_TARGETS[c["lora_targets"]], task_type="CAUSAL_LM")
    mode = TRAINER_MODE.get(reward)
    TrainerCls = PerceptionGRPOTrainer if mode else GRPOTrainer
    extra_kw = dict(mode=mode, c=c) if mode else {}
    trainer = TrainerCls(model=c["model"], args=args, reward_funcs=fns_w, train_dataset=to_dataset(_items or TRAIN),
                         peft_config=peft, processing_class=get_processor(c["model"]), callbacks=[StepCB(gs)], **extra_kw)
    if HAVE_GPU: torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        trainer.train()
    except BaseException:
        trainer = None; gc.collect(); torch.cuda.empty_cache(); raise   # do not keep the failed model on the GPU
    secs = time.time() - t0
    cost = dict(run=name, reward=reward, seed=seed, seconds=secs, gpu_hours=secs/3600,
                peak_gpu_gb=torch.cuda.max_memory_allocated()/2**30 if HAVE_GPU else None,
                steps=c["max_steps"], gpu=torch.cuda.get_device_name(0) if HAVE_GPU else "cpu", **{k: str(v) for k, v in ov.items()})
    trainer.save_model(str(out))
    pd.DataFrame(gs.rows).to_csv(out/"group_stats.csv", index=False)
    if mode: pd.DataFrame(trainer.extra_log).to_csv(out/f"{mode}_terms.csv", index=False)
    if GATE["log"]: pd.DataFrame(GATE["log"]).to_csv(out/"gate_log.csv", index=False)
    pd.DataFrame(trainer.state.log_history).to_csv(out/"log_history.csv", index=False)
    json.dump(cost, open(out/"cost.json", "w"), indent=1); print(cost)
    del trainer; gc.collect(); torch.cuda.empty_cache(); return out

## 8b. PAPO and CFPO loss terms (sharp-gap baselines)
Both are re-implemented as a subclass of `GRPOTrainer`: after the normal GRPO loss, the completion log-probs are computed a second time with a corrupted visual input, and the policy is pushed **away** from that corrupted prediction (it must depend on the image).

* **PAPO** (Wang et al., ICLR 2026): corrupted input = image with a random 60% of patches masked (Qwen: whole 2×2 merged blocks of `pixel_values`; SmolVLM: 32-px squares).
  Loss term: $-\gamma\,\widehat{KL}[\pi_\theta\,\|\,\pi_\theta^{mask}] + \eta_1 \log\pi_\theta + \eta_2 \log\pi_\theta^{mask}$ (double-entropy), with $\widehat{KL} = r - \log r - 1$, $r = \pi_\theta/\pi_\theta^{mask}$ per token.
* **CFPO** (arXiv 2606.23206): counterfactual = the visual tokens most related to the question (cosine with the mean question embedding $> \mu + \lambda\sigma$, $\lambda = 2$) are replaced by the mean visual token. Loss term $-\gamma\,\widehat{KL}_{cf}$, $\gamma = 0.02$.
  *The original CFPO intervenes inside every attention layer on the attention-selected image values; here the same idea is applied once on the visual tokens entering the language model. Call it "CFPO-style" in the paper.*

Per-step KL values are saved (`papo_terms.csv`, `cfpo_terms.csv`) so "KL hacking" (KL exploding while accuracy drops) can be audited.

In [ ]:
def _visual_module(model):
    for n, mod in model.named_modules():
        if n.split(".")[-1] == "visual": return mod
    for n, mod in model.named_modules():
        if n.split(".")[-1] == "connector": return mod
    raise RuntimeError("visual module not found")

def mask_pixels(pv, ratio, merge=2):
    """Random patch masking (PAPO)."""
    pv = pv.clone()
    if pv.dim() == 2:                                   # Qwen2.5-VL: [N_patches, D], merge*merge consecutive rows = 1 visual token
        blk = merge * merge; n = pv.shape[0] // blk
        drop = torch.rand(n, device=pv.device) < ratio
        v = pv[: n * blk].view(n, blk, -1); v[drop] = 0
    else:                                               # Idefics3 / SmolVLM: [B, n_img, C, H, W]
        H, W = pv.shape[-2:]; P = 32; gh, gw = H // P, W // P
        m = torch.rand(*pv.shape[:-3], gh, gw, device=pv.device) < ratio
        m = m.repeat_interleave(P, -2).repeat_interleave(P, -1)
        m = torch.nn.functional.pad(m, (0, W - gw * P, 0, H - gh * P))
        pv = pv.masked_fill(m.unsqueeze(-3), 0)
    return pv

def _replace_salient(x, t, lam):
    if x.shape[-1] != t.shape[-1]: return x, None
    flat = x.reshape(-1, x.shape[-1])
    s = torch.nn.functional.cosine_similarity(flat.float(), t.float()[None], dim=-1)
    m = s > (s.mean() + lam * s.std())
    if m.sum() == 0: m = s >= torch.quantile(s, 0.9)
    new = torch.where(m[:, None], flat.mean(0, keepdim=True).expand_as(flat), flat)
    return new.reshape(x.shape), float(m.float().mean())

def _cf_hook(t, lam, store):
    def hook(mod, inp, out):
        if torch.is_tensor(out):
            new, sh = _replace_salient(out, t, lam); store.append(sh); return new
        if isinstance(out, tuple):
            new, sh = _replace_salient(out[0], t, lam); store.append(sh); return (new,) + tuple(out[1:])
        for k in ("pooler_output", "last_hidden_state"):
            v = getattr(out, k, None)
            if torch.is_tensor(v) and v.shape[-1] == t.shape[-1]:
                new, sh = _replace_salient(v, t, lam); store.append(sh)
                try: out[k] = new
                except Exception: setattr(out, k, new)
                return out
        return out
    return hook

def _question_vec(model, prompt_ids, prompt_mask):
    E = model.get_input_embeddings()(prompt_ids)
    vecs = [e[m.bool()][-45:-5].float().mean(0) for e, m in zip(E, prompt_mask)]
    return torch.stack(vecs).mean(0).detach()

class PerceptionGRPOTrainer(GRPOTrainer):
    def __init__(self, *a, mode=None, c=None, **kw):
        super().__init__(*a, **kw); self.mode, self.c, self.extra_log, self._warned = mode, c, [], False

    def _completion_logps(self, model, inputs, pixel_values):
        ids = torch.cat([inputs["prompt_ids"], inputs["completion_ids"]], 1)
        am = torch.cat([inputs["prompt_mask"], inputs["completion_mask"]], 1)
        kw = {k: inputs[k] for k in ("image_grid_thw", "pixel_attention_mask", "image_sizes") if inputs.get(k) is not None}
        L = inputs["completion_ids"].shape[1]
        logits = model(input_ids=ids, attention_mask=am, pixel_values=pixel_values, **kw).logits[:, -L - 1:-1].float()
        return torch.log_softmax(logits, -1).gather(-1, inputs["completion_ids"].unsqueeze(-1)).squeeze(-1)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        res = super().compute_loss(model, inputs, return_outputs=return_outputs, num_items_in_batch=num_items_in_batch)
        loss, rest = (res[0], res[1:]) if isinstance(res, tuple) else (res, None)
        need = ("prompt_ids", "prompt_mask", "completion_ids", "completion_mask", "pixel_values")
        if self.mode is None or any(inputs.get(k) is None for k in need):
            if not self._warned: print("PerceptionGRPOTrainer: missing keys, extra term skipped. Keys =", list(inputs)); self._warned = True
            return res
        cm = inputs["completion_mask"].float(); den = cm.sum().clamp(min=1); pv = inputs["pixel_values"]
        lp = self._completion_logps(model, inputs, pv); store = []
        if self.mode == "papo":
            lp_cf = self._completion_logps(model, inputs, mask_pixels(pv, self.c["papo_mask_ratio"])); gamma = self.c["papo_gamma"]
        else:
            h = _visual_module(model).register_forward_hook(_cf_hook(_question_vec(model, inputs["prompt_ids"], inputs["prompt_mask"]), self.c["cfpo_lambda"], store))
            try: lp_cf = self._completion_logps(model, inputs, pv)
            finally: h.remove()
            gamma = self.c["cfpo_gamma"]
        logr = (lp - lp_cf).clamp(-10, 10)
        kl = ((torch.exp(logr) - logr - 1) * cm).sum() / den          # r - log r - 1, r = pi / pi_corrupted
        extra = -gamma * kl                                            # maximise the KL
        if self.mode == "papo":                                        # double entropy: J -= eta1*log pi + eta2*log pi_mask
            extra = extra + self.c["papo_eta1"] * (lp * cm).sum() / den + self.c["papo_eta2"] * (lp_cf * cm).sum() / den
        self.extra_log.append(dict(step=self.state.global_step, kl=float(kl.detach()), extra=float(extra.detach()),
                                   salient_share=(float(np.mean([x for x in store if x is not None])) if any(x is not None for x in store) else None)))
        loss = loss + extra
        return (loss, *rest) if rest is not None else loss

## 9. Evaluation: accuracy, hallucination in reasoning (G5), POPE, reward-hacking audit (C)
* **conflict_acc** (says `None` when object absent) · **normal_acc** · **false_none** (over-refusal) · **format_rate**
* **CHAIR_i** (object-level, on `<think>`): claimed-but-absent objects / all claimed objects. **CHAIR_s**: share of answers with ≥1 hallucinated object. Evaluation uses COCO annotations — allowed, because it is evaluation only.
* **Image dependence:** accuracy and None-rate with a grey image instead of the real one. A model that truly looks should answer `None` almost always on the grey image (idea C, counterfactual).
* **Counterfactual reasoning faithfulness:** on the grey image, the *reasoning* should not claim to see any object. `blank_reasoning_hall` = share of answers whose `<think>` still claims objects on a grey image (pure language-prior hallucination in the chain of thought).
* **Goodhart gap:** final training reward − held-out accuracy; **copy overlap**; reasoning length.

In [ ]:
def eval_items(items, outputs):
    rows = []
    for x, o in zip(items, outputs):
        th, an = parse(o); a = norm_ans(an); g = norm_ans(x["solution"])
        claimed = affirmed_objects(th or ""); hall = claimed - set(x["present"])
        w = (th or "").lower().split(); q = set(x["question"].lower().split())
        rows.append(dict(kind=x["kind"], correct=int(a == g), formatted=int(th is not None and an is not None),
                         said_none=int(a == "none"), false_none=int(x["kind"] == "normal" and a == "none"),
                         n_claim=len(claimed), n_hall=len(hall), any_hall=int(len(hall) > 0),
                         asked_claimed=int(x["asked"] in claimed), think_len=len(w),
                         copy=float(np.mean([t in q for t in w])) if w else np.nan,
                         abs_err=abs(int(a) - int(g)) if (a and g and a.isdigit() and g.isdigit()) else np.nan))
    return pd.DataFrame(rows)

def summarize(df):
    c, n = df[df.kind == "conflict"], df[df.kind == "normal"]
    return dict(acc=df.correct.mean(), conflict_acc=c.correct.mean(), normal_acc=n.correct.mean(),
                false_none=n.false_none.mean(), none_rate=df.said_none.mean(), format_rate=df.formatted.mean(),
                chair_i=df.n_hall.sum() / max(1, df.n_claim.sum()), chair_s=df.any_hall.mean(),
                conflict_asked_claimed=c.asked_claimed.mean(), think_len=df.think_len.mean(),
                copy=df["copy"].mean(), mae_count=n.abs_err.mean())

def blank_test(model, proc, items):
    items = items[:CFG["blank_n"]]; outs = generate(model, proc, items, blank=True)
    ans = [norm_ans(parse(o)[1]) for o in outs]
    normal = [i for i, x in enumerate(items) if x["kind"] == "normal"]
    claims = [affirmed_objects(parse(o)[0] or "") for o in outs]          # objects the reasoning claims to SEE on a plain grey image
    json.dump(outs, open(RES/"last_blank_outputs.json", "w"))
    return dict(blank_none_rate=float(np.mean([a == "none" for a in ans])),
                blank_reasoning_hall=float(np.mean([len(c) > 0 for c in claims])),   # CoT counterfactual faithfulness: should be ~0
                blank_claims_per_answer=float(np.mean([len(c) for c in claims])),
                blank_normal_acc=float(np.mean([ans[i] == norm_ans(items[i]["solution"]) for i in normal])) if normal else np.nan,
                blank_outcome_reward=float(np.mean(r_outcome(outs, [x["solution"] for x in items]))))

def load_pope(n):
    from datasets import load_dataset
    d = load_dataset("lmms-lab/POPE", split="test")
    cols = d.column_names; print("POPE columns:", cols)
    return d.shuffle(seed=CFG["seed"]).select(range(min(n, len(d))))

@torch.no_grad()
def eval_pope(model, proc, n=None):
    d = load_pope(n or CFG["pope_n"]); recs = []
    for i in range(0, len(d), 8):
        b = [d[j] for j in range(i, min(i + 8, len(d)))]
        msgs = [[{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": x["question"] + " Answer with Yes or No only."}]}] for x in b]
        inp = proc_call(proc, [proc.apply_chat_template(m, add_generation_prompt=True) for m in msgs],
                        [load_img_pil(x["image"]) for x in b]).to(model.device)
        g = model.generate(**inp, max_new_tokens=4, do_sample=False)
        for x, o in zip(b, proc.batch_decode(g[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)):
            pred = "yes" if o.strip().lower().startswith("yes") else "no"; gold = str(x["answer"]).strip().lower()
            recs.append(dict(category=x.get("category", "all"), pred=pred, gold=gold, correct=int(pred == gold)))
    df = pd.DataFrame(recs); tp = ((df.pred == "yes") & (df.gold == "yes")).sum()
    prec = tp / max(1, (df.pred == "yes").sum()); rec = tp / max(1, (df.gold == "yes").sum())
    return dict(pope_acc=df.correct.mean(), pope_f1=2*prec*rec/max(1e-9, prec+rec), pope_yes_ratio=(df.pred == "yes").mean()), df

def load_img_pil(im, side=None):
    im = im.convert("RGB"); side = side or CFG["max_img_side"]
    if max(im.size) > side:
        r = side / max(im.size); im = im.resize((int(im.width*r), int(im.height*r)))
    return im

## 10. Coverage–risk curve and AURC (idea I, abstention quality)
Confidence = agreement of k sampled answers with their majority. Sort by confidence; risk at coverage c = error rate of the c most confident answers. Lower AURC = the model knows when it is wrong.

In [ ]:
def coverage_risk(model, proc, items):
    items = items[:CFG["cov_risk_n"]]; k = CFG["cov_risk_k"]
    samples = generate(model, proc, items, batch=4, sample=True, n=k)
    conf, corr = [], []
    for x, ss in zip(items, samples):
        ans = [norm_ans(parse(s)[1]) for s in ss]; maj, cnt = collections.Counter(ans).most_common(1)[0]
        conf.append(cnt / k); corr.append(int(maj == norm_ans(x["solution"])))
    o = np.argsort(-np.array(conf), kind="stable"); err = 1 - np.array(corr)[o]
    cov = np.arange(1, len(o) + 1) / len(o); risk = np.cumsum(err) / np.arange(1, len(o) + 1)
    return dict(aurc=float((getattr(np, "trapezoid", None) or np.trapz)(risk, cov)) if len(o) > 1 else float(risk.mean())), pd.DataFrame(dict(coverage=cov, risk=risk))

## 10b. Full evaluation of one model (writes everything to the registry)

In [ ]:
def starvation_curve(run_dir):
    """Per-step share of zero-variance GRPO groups (total reward), from TRL's own log."""
    f = Path(run_dir)/"log_history.csv"
    if not f.exists(): return None
    lh = pd.read_csv(f)
    if "frac_reward_zero_std" not in lh or "step" not in lh: return None
    s_ = lh.dropna(subset=["frac_reward_zero_std"]).groupby("step").frac_reward_zero_std.mean()
    return s_ if len(s_) else None

def registry():
    return [json.loads(l) for l in open(REGISTRY)] if REGISTRY.exists() else []
def already(name): return any(r["run"] == name for r in registry())

def evaluate_run(name, adapter=None, extras=True, meta=None, model=None):
    if already(name): print("skip eval (exists)", name); return
    d = RUNS/name; d.mkdir(exist_ok=True, parents=True)
    m, p = load_policy(adapter, model=model)
    try:
        rec = _evaluate(name, d, m, p, extras, meta)
    finally:                                   # free the GPU even if an evaluation step fails
        del m; gc.collect(); torch.cuda.empty_cache()
    print(json.dumps(rec, indent=1, default=str))

def _evaluate(name, d, m, p, extras, meta):
    out = generate(m, p, TEST); df = eval_items(TEST, out)
    df.to_csv(d/"eval_items.csv", index=False); json.dump(out, open(d/"outputs.json", "w"))
    rec = dict(run=name, **(meta or {}), **summarize(df), **blank_test(m, p, TEST))
    if extras:
        pope, pdf = eval_pope(m, p); pdf.to_csv(d/"pope.csv", index=False); rec.update(pope)
        cr, crdf = coverage_risk(m, p, TEST); crdf.to_csv(d/"cov_risk.csv", index=False); rec.update(cr)
    if (d/"cost.json").exists(): rec.update({k: v for k, v in json.load(open(d/"cost.json")).items() if k in ("seconds", "gpu_hours", "peak_gpu_gb")})
    if (d/"log_history.csv").exists():
        lh = pd.read_csv(d/"log_history.csv"); rc = [c for c in lh.columns if c == "reward"] or [c for c in lh.columns if c.startswith("reward") and "std" not in c]
        rec["final_train_reward"] = float(lh[rc[0]].dropna().tail(10).mean()) if rc else np.nan
        if "frac_reward_zero_std" in lh:          # share of GRPO groups whose TOTAL reward has zero variance (no gradient)
            rec["zero_std_share"] = rec["trl_zero_std"] = float(lh["frac_reward_zero_std"].dropna().mean())
    if (d/"group_stats.csv").exists():            # share of groups starved by the OUTCOME reward alone (where A-gated opens)
        g = pd.read_csv(d/"group_stats.csv"); g = g[g.reward.isin(["r_outcome", "r_mcr1"])]
        if len(g): rec["outcome_zero_std"] = float(g.zero_std_share.mean())
    with open(REGISTRY, "a") as f: f.write(json.dumps({k: (None if isinstance(v, float) and np.isnan(v) else v) for k, v in rec.items()}) + "\n")
    return rec

## 11b. Statistics (E): mean ± sd over seeds, paired bootstrap CI, exact McNemar, Holm correction

In [ ]:
from scipy.stats import binomtest
def paired_bootstrap(a, b, n=None, seed=0):
    a, b = np.asarray(a, float), np.asarray(b, float); n = n or CFG["bootstrap_n"]; rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), (n, len(a))); d = (b[idx] - a[idx]).mean(1)
    return float((b - a).mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))
def mcnemar_exact(a, b):
    a, b = np.asarray(a), np.asarray(b); n01 = int(((a == 0) & (b == 1)).sum()); n10 = int(((a == 1) & (b == 0)).sum())
    return n01, n10, (1.0 if n01 + n10 == 0 else float(binomtest(n01, n01 + n10, 0.5).pvalue))
def holm(ps):
    ps = np.asarray(ps, float); o = np.argsort(ps); adj = np.empty_like(ps); m = len(ps); run = 0
    for r, i in enumerate(o): run = max(run, min(1, (m - r) * ps[i])); adj[i] = run
    return adj

def compare_to(ref_run, runs, col="correct", subset=None):
    ra = pd.read_csv(RUNS/ref_run/"eval_items.csv"); rows = []
    for r in runs:
        f = RUNS/r/"eval_items.csv"
        if not f.exists() or r == ref_run: continue
        rb = pd.read_csv(f); a, b = ra, rb
        if subset: a, b = ra[ra.kind == subset], rb[rb.kind == subset]
        diff, lo, hi = paired_bootstrap(a[col], b[col]); n01, n10, p = mcnemar_exact(a[col], b[col])
        rows.append(dict(run=r, vs=ref_run, metric=col + (f"[{subset}]" if subset else ""), diff=diff, ci_lo=lo, ci_hi=hi, b_only=n01, a_only=n10, p=p))
    df = pd.DataFrame(rows)
    if len(df): df["p_holm"] = holm(df.p)
    return df

## C1. Comparison set-up

In [ ]:
CMP = RES/"comparison"; CTAB = CMP/"tables"; CFIG = CMP/"figures"
for d in (CMP, CTAB, CFIG): d.mkdir(parents=True, exist_ok=True)
REGISTRY = RES/"registry.jsonl"          # reuse the part 1 evaluations (no re-evaluation of the same models)
CFG["mmmc_n"] = 200
if CFG["preset"] == "smoke": CFG["mmmc_n"] = 12
METHODS = {                    # display name -> (reward set, source)
    "Zero-shot":            (None, "Qwen2.5-VL-3B-Instruct"),
    "GRPO (outcome)":       ("outcome", "vanilla RLVR"),
    "MC-R1":                ("mcr1", "Chen et al., IEEE TMM 2026"),
    "PAPO":                 ("papo_loss", "Wang et al., ICLR 2026"),
    "CFPO":                 ("cfpo", "arXiv 2606.23206"),
    "Ours (outcome+A-gated)": ("outcome_A_gated", "this work"),
}
OURS = "Ours (outcome+A-gated)"
SEEDS = CFG["seeds"]
def mrun(method, seed):
    rw = METHODS[method][0]
    return "zero_shot" if rw is None else run_name(rw, seed)
PROPS = pd.DataFrame([
    dict(method="MC-R1", changes="reward (3 string-match terms)", gold_answer="yes", annotations="no", judge_or_RM="no", extra_policy_forward=0, extra_model="spaCy parser"),
    dict(method="PAPO", changes="loss (masked-image KL + double entropy)", gold_answer="yes (verifier)", annotations="no", judge_or_RM="no", extra_policy_forward=1, extra_model="none"),
    dict(method="CFPO", changes="loss (counterfactual KL)", gold_answer="yes (verifier)", annotations="no", judge_or_RM="no", extra_policy_forward=1, extra_model="none"),
    dict(method=OURS, changes="reward (consistency A, gated)", gold_answer="only for the gate", annotations="no", judge_or_RM="no", extra_policy_forward=0, extra_model="frozen CLIP (image features cached)"),
])
FAIR = pd.DataFrame([dict(setting=k, value=str(CFG[k])) for k in ("model", "n_train", "n_test", "max_steps", "lr", "num_generations", "per_device_bs", "grad_accum", "beta", "lora_r", "lora_targets", "max_completion_len", "seeds")])
FAIR.to_csv(CMP/"fair_protocol.csv", index=False); PROPS.to_csv(CMP/"method_properties.csv", index=False)
print(FAIR.to_string(index=False))

## C2. Train every method with the same budget and evaluate in-domain (resume-safe; runs from the main notebook are reused)

In [ ]:
for method, (rw, _) in METHODS.items():
    for seed in ([0] if rw is None else SEEDS):
        name = mrun(method, seed)
        try:
            ad = None
            if rw is not None: ad = str(train(rw, seed))
            evaluate_run(name, adapter=ad, extras=(seed == SEEDS[0] or rw is None), meta=dict(method=method, seed=seed, reward=rw or "zero_shot"))
        except Exception as e:
            print("FAILED", method, seed, repr(e)); open(CMP/"failed.txt", "a").write(f"{method} {seed} {e!r}\n")

## C3. Out-of-domain: MMMC test (MC-R1's own setting)
Every MMMC question contains a modality conflict (object, attribute or relationship that is not in the image). Metrics: ROUGE-L and weighted ROUGE-L / KTOR against the reference answer (as in MMMC / MC-R1), and **Hallu-rate** — share of responses that assume something not in the image — judged by a separate **Qwen2.5-VL-7B** judge (the MMMC paper used GPT-4o / Llama-3.3-70B; we use an open model so the result is reproducible; state this). All answers are generated first; the judge is loaded once afterwards.

In [ ]:
SYSTEM_OPEN = ("You are a careful visual assistant. First check inside <think></think> what is actually visible in the image. "
               "Then answer inside <answer></answer>. If the question assumes an object, attribute or relation that is not in the image, say so.")
def load_mmmc(n):
    from datasets import load_dataset
    for rid in ("MM-Hallu/MMMC", "ustc-zhangzm/MMMC"):
        try: d = load_dataset(rid, split="test"); print("MMMC from", rid, d.column_names); break
        except Exception as e: print("could not load", rid, e); d = None
    if d is None: return []
    d = d.shuffle(seed=CFG["seed"]).select(range(min(n, len(d)))); out = []
    idir = ROOT/"mmmc_images"; idir.mkdir(exist_ok=True)
    for i, x in enumerate(d):
        f = idir/f"{i}.jpg"
        if not f.exists(): x["image"].convert("RGB").save(f)
        out.append(dict(image=str(f), question=x["question"], answer=x["answer"], conflict_type=x.get("conflict_type", "")))
    return out
MMMC = load_mmmc(CFG["mmmc_n"]); print(len(MMMC), "MMMC items", collections.Counter(x["conflict_type"] for x in MMMC))

def rouge_l(pred, ref):
    a, b = re.findall(r"\w+", (pred or "").lower()), re.findall(r"\w+", (ref or "").lower())
    if not a or not b: return 0.0
    D = np.zeros((len(a) + 1, len(b) + 1))
    for i in range(len(a)):
        for j in range(len(b)): D[i+1, j+1] = D[i, j] + 1 if a[i] == b[j] else max(D[i, j+1], D[i+1, j])
    l = D[-1, -1]; p, r = l / len(a), l / len(b); return 0.0 if l == 0 else 2 * p * r / (p + r)

def mmmc_generate(name, adapter):
    f = RUNS/name/"mmmc_outputs.json"
    if f.exists(): return json.load(open(f))
    (RUNS/name).mkdir(parents=True, exist_ok=True)
    m, p = load_policy(adapter); outs = generate(m, p, MMMC, system=SYSTEM_OPEN)
    json.dump(outs, open(f, "w")); del m; gc.collect(); torch.cuda.empty_cache(); return outs

HALLU_Q = ("Question asked about this image: \"{q}\"\nResponse: \"{r}\"\nDoes the response assume or describe any object, attribute "
           "or relationship that is NOT actually in the image (instead of pointing out that it is not there)? Answer Yes or No.")
@torch.no_grad()
def judge_hallu(items, outs):
    m, p = get_judge(); yes = p.tokenizer.encode("Yes", add_special_tokens=False)[0]; no = p.tokenizer.encode("No", add_special_tokens=False)[0]; r = []
    for x, o in zip(items, outs):
        ans = parse(o)[1] or o
        msg = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": HALLU_Q.format(q=x["question"], r=ans[:800])}]}]
        z = proc_call(p, [p.apply_chat_template(msg, add_generation_prompt=True)], [load_img(x["image"])]).to(m.device)
        lg = m(**z).logits[0, -1].float(); r.append(float(torch.softmax(lg[[yes, no]], 0)[0]))
    return np.array(r)

MM = {}
if MMMC:
    for method, (rw, _) in METHODS.items():
        for seed in ([0] if rw is None else SEEDS):
            name = mrun(method, seed); ad = None if rw is None else RUNS/name
            if ad is not None and not (ad/"adapter_model.safetensors").exists(): continue
            MM[(method, seed)] = mmmc_generate(name, None if ad is None else str(ad))
    for (method, seed), outs in MM.items():                # judge loaded once, after all generations
        f = RUNS/mrun(method, seed)/"mmmc_eval.csv"
        if f.exists(): continue
        h = judge_hallu(MMMC, outs); rows = []
        for x, o, hp in zip(MMMC, outs, h):
            a = parse(o)[1] or o
            rows.append(dict(conflict_type=x["conflict_type"], rouge_l=rouge_l(a, x["answer"]), w_rouge_l=weighted_rouge_l(a, x["answer"]),
                             ktor=ktor(a, x["answer"]), hallu=int(hp > 0.5), hallu_p=hp,
                             flags_conflict=int(bool(NEG.search(a.lower())) or norm_ans(a) == "none")))
        pd.DataFrame(rows).to_csv(f, index=False)

## C4. Comparison table (mean ± sd over seeds)

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 8, "font.family": "serif", "axes.spines.top": False, "axes.spines.right": False})
COLW = 3.45
def to_latex(df, path, caption, label, fmt=3):
    d = df.copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]): d[c] = d[c].map(lambda v: "--" if pd.isna(v) else f"{v:.{fmt}f}")
    body = d.to_latex(index=False, escape=False, column_format="l" + "c" * (len(d.columns) - 1)).replace("_", r"\_").replace("±", r"$\pm$").replace("−", "w/o ")
    open(path, "w").write("\\begin{table}[t]\n\\centering\n\\caption{%s}\n\\label{%s}\n\\resizebox{\\columnwidth}{!}{%%\n%s}\n\\end{table}\n" % (caption, label, body))
def savefig(fig, name): fig.tight_layout(); fig.savefig(CFIG/f"{name}.png", dpi=400); fig.savefig(CFIG/f"{name}.pdf"); plt.close(fig)

# metric -> (label, higher is better?)
METRICS = {"acc": ("COCO acc", True), "conflict_acc": ("conflict acc", True), "false_none": ("false-None", False),
           "chair_i": ("CHAIR$_i$", False), "blank_reasoning_hall": ("grey-img hallu", False),
           "pope_f1": ("POPE F1", True), "aurc": ("AURC", False),
           "mmmc_rouge_l": ("MMMC ROUGE-L", True), "mmmc_hallu": ("MMMC hallu", False),
           "zero_std_share": ("zero-var groups", False), "gpu_hours": ("GPU h", False), "peak_gpu_gb": ("peak GB", False)}
def per_run_rows():
    reg = pd.DataFrame(registry()); rows = []
    for method, (rw, _) in METHODS.items():
        for seed in ([0] if rw is None else SEEDS):
            name = mrun(method, seed); r = reg[reg.run == name] if len(reg) else pd.DataFrame()
            row = dict(method=method, seed=seed, run=name)
            if len(r): row.update({k: r.iloc[-1][k] for k in METRICS if k in r.columns})
            f = RUNS/name/"mmmc_eval.csv"
            if f.exists():
                mm = pd.read_csv(f); row.update(mmmc_rouge_l=mm.rouge_l.mean(), mmmc_w_rouge_l=mm.w_rouge_l.mean(), mmmc_ktor=mm.ktor.mean(),
                                                mmmc_hallu=mm.hallu.mean(), mmmc_flags_conflict=mm.flags_conflict.mean())
            if len(row) > 3: rows.append(row)
    return pd.DataFrame(rows)
RUNS_DF = per_run_rows(); RUNS_DF.to_csv(CMP/"per_run.csv", index=False)
def mean_sd(df):
    out = []
    for method in METHODS:
        s = df[df.method == method]
        if not len(s): continue
        row = dict(method=method, n_seeds=len(s))
        for k in METRICS:
            if k in s:
                v = pd.to_numeric(s[k], errors="coerce").dropna()
                row[k] = v.mean() if len(v) else np.nan; row[k + "_sd"] = v.std() if len(v) > 1 else np.nan
        out.append(row)
    return pd.DataFrame(out)
CT = mean_sd(RUNS_DF) if len(RUNS_DF) else pd.DataFrame()
if len(CT):
    show = CT[["method", "n_seeds"]].copy()
    for k, (lab, _) in METRICS.items():
        if k in CT: show[lab] = [("--" if pd.isna(m) else (f"{m:.3f}" + ("" if pd.isna(s) else f" ± {s:.3f}"))) for m, s in zip(CT[k], CT.get(k + "_sd", pd.Series([np.nan] * len(CT))))]
    show.to_csv(CTAB/"comparison_main.csv", index=False)
    to_latex(show, CTAB/"comparison_main.tex", "Base papers vs. our method under the same budget (mean $\\pm$ sd over seeds). Arrows: see text.", "tab:cmp_main")
    to_latex(PROPS, CTAB/"comparison_properties.tex", "What each method changes and needs.", "tab:cmp_props")
    to_latex(FAIR, CTAB/"comparison_protocol.tex", "Shared training protocol for all methods.", "tab:cmp_protocol")
show if len(CT) else None

## C5. Significance: ours vs each baseline (paired over the same test items, all seeds stacked)

In [ ]:
def item_vec(method, col, source="coco", kind=None):
    parts = []
    for seed in ([0] if METHODS[method][0] is None else (SEEDS[:1] if source == "pope" else SEEDS)):
        f = RUNS/mrun(method, seed)/("eval_items.csv" if source == "coco" else "mmmc_eval.csv" if source == "mmmc" else "pope.csv")
        if not f.exists(): return None
        d = pd.read_csv(f)
        if kind: d = d[d.kind == kind]
        parts.append(d[col].to_numpy())
    return parts
TESTS = [("COCO accuracy", "coco", "correct", None, True), ("COCO conflict accuracy", "coco", "correct", "conflict", True),
         ("reasoning hallucination (any)", "coco", "any_hall", None, False), ("MMMC hallucination", "mmmc", "hallu", None, False),
         ("POPE accuracy", "pope", "correct", None, True)]
sig = []
for base in [m for m in METHODS if m != OURS]:
    for lab, src, col, kind, hib in TESTS:
        a, b = item_vec(base, col, src, kind), item_vec(OURS, col, src, kind)
        if not a or not b: continue
        if len(a) == 1 and len(b) > 1: a = a * len(b)                # zero-shot has one run: pair it with every seed of ours
        k = min(len(a), len(b)); a, b = a[:k], b[:k]
        a, b = np.concatenate(a), np.concatenate(b)
        if len(a) != len(b): continue
        d, lo, hi = paired_bootstrap(a, b); n01, n10, p = mcnemar_exact(a, b)
        sig.append(dict(baseline=base, metric=lab, higher_better=hib, ours_minus_base=d, ci_lo=lo, ci_hi=hi, ours_only=n01, base_only=n10, p=p))
SIG = pd.DataFrame(sig)
if len(SIG):
    SIG["p_holm"] = holm(SIG.p.to_numpy())
    def verdict(r):
        better = r.ours_minus_base > 0 if r.higher_better else r.ours_minus_base < 0
        if abs(r.ours_minus_base) < 1e-9: return "tie"
        return ("ours better" if better else "ours worse") + (" (sig.)" if r.p_holm < 0.05 else " (n.s.)")
    SIG["verdict"] = SIG.apply(verdict, axis=1); SIG.to_csv(CTAB/"comparison_significance.csv", index=False)
    to_latex(SIG.drop(columns=["higher_better", "p"]), CTAB/"comparison_significance.tex",
             "Ours minus baseline on the same items (all seeds): paired bootstrap 95\\% CI and exact McNemar test, Holm-adjusted.", "tab:cmp_sig")
SIG

## C6. Verdict: where ours is better, where it is not, and why
The table counts wins/ties/losses from the significance tests. The "why" lines are generated **only from measured evidence** (gradient signal, grounding of the reasoning, image dependence, cost, requirements) — copy the ones that hold into the Discussion section and explain them in your own words.

In [ ]:
if len(SIG):
    WTL = SIG.groupby("baseline").verdict.value_counts().unstack(fill_value=0); WTL.to_csv(CTAB/"win_tie_loss.csv"); print(WTL, "\n")
why = []
if len(CT) and OURS in set(CT.method):
    o = CT.set_index("method").loc[OURS]
    for b in [m for m in CT.method if m not in (OURS, "Zero-shot")]:
        x = CT.set_index("method").loc[b]
        def cmp_(k, hib):
            if k not in CT or pd.isna(o.get(k)) or pd.isna(x.get(k)): return None
            return (o[k] > x[k]) if hib else (o[k] < x[k])
        if cmp_("zero_std_share", False): why.append(f"vs {b}: fewer zero-variance GRPO groups ({o.zero_std_share:.2f} vs {x.zero_std_share:.2f}) — the gate gives a learning signal exactly where the outcome reward gives none.")
        if cmp_("chair_i", False): why.append(f"vs {b}: less hallucination inside the reasoning (CHAIR_i {o.chair_i:.3f} vs {x.chair_i:.3f}) — A rewards reasoning whose claimed objects CLIP can verify.")
        if cmp_("blank_reasoning_hall", False): why.append(f"vs {b}: on a grey image the reasoning claims fewer objects ({o.blank_reasoning_hall:.2f} vs {x.blank_reasoning_hall:.2f}) — less language-prior hallucination.")
        if cmp_("mmmc_hallu", False): why.append(f"vs {b}: lower out-of-domain hallucination on MMMC ({o.mmmc_hallu:.3f} vs {x.mmmc_hallu:.3f}).")
        if cmp_("gpu_hours", False): why.append(f"vs {b}: cheaper training ({o.gpu_hours:.2f} vs {x.gpu_hours:.2f} GPU h) — no extra policy forward pass, no judge.")
        for k, lab, hib in (("acc", "COCO accuracy", True), ("conflict_acc", "conflict accuracy", True), ("mmmc_hallu", "MMMC hallucination", False)):
            c_ = cmp_(k, hib)
            if c_ is False: why.append(f"vs {b}: NOT better on {lab} ({o[k]:.3f} vs {x[k]:.3f}) — report this honestly and discuss it.")
why.append("By design: ours needs no judge, no reward model, no object annotations, and no extra forward pass of the policy (PAPO and CFPO need one; MC-R1 needs reference answers for all three terms).")
open(CMP/"why_ours.txt", "w").write("\n".join(why)); print("\n".join("• " + w for w in why))

## C7. Figures (400 DPI)

In [ ]:
if len(CT):
    order = [m for m in METHODS if m in set(CT.method)]; T = CT.set_index("method").loc[order]
    cols = ["#9aa0a6", "#5f6368", "#1a73e8", "#e37400", "#188038", "#d93025"][:len(order)]
    # Fig 1: key metrics side by side
    keys = [k for k in ("acc", "conflict_acc", "chair_i", "blank_reasoning_hall", "mmmc_hallu", "pope_f1") if k in T]
    fig, axs = plt.subplots(1, len(keys), figsize=(COLW * 2, 2.3))
    for ax, k in zip(np.atleast_1d(axs), keys):
        ax.bar(range(len(order)), T[k], yerr=T.get(k + "_sd"), color=cols, capsize=2)
        ax.set_title(METRICS[k][0] + (" ↑" if METRICS[k][1] else " ↓"), fontsize=7); ax.set_xticks([])
    fig.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in cols], order, ncol=3, fontsize=6, frameon=False, loc="lower center", bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.12, 1, 1)); fig.savefig(CFIG/"fig_cmp_metrics.png", dpi=400); fig.savefig(CFIG/"fig_cmp_metrics.pdf"); plt.close(fig)

    # Fig 2: radar of normalised metrics (1 = best method on that metric)
    rk = [k for k in ("acc", "conflict_acc", "chair_i", "blank_reasoning_hall", "pope_f1", "mmmc_hallu", "aurc", "gpu_hours") if k in T and T[k].notna().sum() > 1]
    if len(rk) >= 3:
        N = np.array([(T[k] - T[k].min()) / (T[k].max() - T[k].min() + 1e-12) if METRICS[k][1] else (T[k].max() - T[k]) / (T[k].max() - T[k].min() + 1e-12) for k in rk]).T
        ang = np.linspace(0, 2 * np.pi, len(rk), endpoint=False).tolist(); ang += ang[:1]
        fig = plt.figure(figsize=(COLW, COLW)); ax = fig.add_subplot(111, polar=True)
        for i, mth in enumerate(order):
            v = np.nan_to_num(N[i]).tolist(); v += v[:1]; ax.plot(ang, v, color=cols[i], lw=1.4 if mth == OURS else 0.8, label=mth)
            if mth == OURS: ax.fill(ang, v, color=cols[i], alpha=0.15)
        ax.set_xticks(ang[:-1]); ax.set_xticklabels([METRICS[k][0] for k in rk], fontsize=6); ax.set_yticklabels([])
        ax.legend(fontsize=5, frameon=False, loc="upper right", bbox_to_anchor=(1.35, 1.12)); savefig(fig, "fig_cmp_radar")

    # Fig 3: cost vs accuracy (Pareto)
    if "gpu_hours" in T and T.gpu_hours.notna().any():
        fig, ax = plt.subplots(figsize=(COLW, 2.4))
        for i, mth in enumerate(order):
            if pd.notna(T.loc[mth].get("gpu_hours")): ax.scatter(T.loc[mth, "gpu_hours"], T.loc[mth, "acc"], color=cols[i], s=40 if mth == OURS else 18); ax.annotate(mth, (T.loc[mth, "gpu_hours"], T.loc[mth, "acc"]), fontsize=5)
        ax.set_xlabel("GPU hours per run"); ax.set_ylabel("COCO accuracy"); savefig(fig, "fig_cmp_cost_accuracy")

# Fig 4: gradient starvation per method (seed 42)
fig, ax = plt.subplots(figsize=(COLW, 2.2)); any_ = False
for mth, (rw, _) in METHODS.items():
    s_ = starvation_curve(RUNS/mrun(mth, SEEDS[0])) if rw else None
    if s_ is not None: ax.plot(s_.index, s_.rolling(5, min_periods=1).mean(), label=mth); any_ = True
if any_: ax.set_xlabel("training step"); ax.set_ylabel("zero-variance groups"); ax.legend(fontsize=5, frameon=False); savefig(fig, "fig_cmp_starvation")
else: plt.close(fig)

# Fig 5: MMMC hallucination by conflict type
rows = []
for mth, (rw, _) in METHODS.items():
    f = RUNS/mrun(mth, SEEDS[0] if rw else 0)/"mmmc_eval.csv"
    if f.exists():
        d = pd.read_csv(f); rows += [dict(method=mth, conflict_type=t, hallu=v) for t, v in d.groupby("conflict_type").hallu.mean().items()]
if rows:
    P = pd.DataFrame(rows).pivot(index="conflict_type", columns="method", values="hallu")[[m for m in METHODS if m in set(r["method"] for r in rows)]]
    P.to_csv(CTAB/"mmmc_by_type.csv"); ax = P.plot.bar(figsize=(COLW * 2, 2.3), rot=0); ax.set_ylabel("MMMC Hallu-rate ↓"); ax.legend(fontsize=6, frameon=False)
    savefig(ax.get_figure(), "fig_cmp_mmmc_types")
print(sorted(p.name for p in CFIG.iterdir()))

## C8. Package

In [ ]:
zp = shutil.make_archive(str(ROOT/"A3_comparison_results"), "zip", CMP); print(zp)
try:
    from google.colab import files; files.download(zp)
except Exception: pass